Connected to media-mix-modeling (Python 3.12.2)

 # Phase 1: EDA and Setup
 Load raw Conjura data, aggregate to df_weekly, profile brands, select modeling candidate.

In [ ]:
import os

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import statsmodels.api as sm
from plotly.subplots import make_subplots

df = pd.read_csv("data/raw/conjura_mmm_data.csv")
df.columns = df.columns.str.lower()
df["date_day"] = pd.to_datetime(df["date_day"])
mask = df["territory_name"] == "US"
df = df[mask]
print(f"Shape: {df.shape}")

Shape: (9089, 50)


In [ ]:
cols_spend = [c for c in df.columns if c.endswith("_spend")]

df_grain = (
    df.groupby("organisation_id")
    .agg(
        vertical=("organisation_vertical", "first"),
        date_min=("date_day", "min"),
        date_max=("date_day", "max"),
        n_days=("date_day", "nunique"),
        # all purchases
        mean_all_purchases=("all_purchases", "mean"),
        std_all_purchases=("all_purchases", "std"),
        mean_all_purchases_units=("all_purchases_units", "mean"),
        std_all_purchases_units=("all_purchases_units", "std"),
        # first purchases
        mean_first_purchases=("first_purchases", "mean"),
        std_first_purchases=("first_purchases", "std"),
        mean_first_purchases_units=("first_purchases_units", "mean"),
        std_first_purchases_units=("first_purchases_units", "std"),
    )
)

# Compute spend and channel counts from the grouped data
for org_id in df_grain.index:
    org_data = df.loc[df["organisation_id"] == org_id, cols_spend]
    df_grain.loc[org_id, "total_spend"] = org_data.sum().sum()
    df_grain.loc[org_id, "n_active_channels"] = (org_data.sum() > 0).sum()

df_grain = df_grain.sort_values("total_spend", ascending=False)

print(f"US brands: {len(df_grain)}")
df_grain.head(100)

US brands: 10


,vertical,date_min,date_max,n_days,mean_all_purchases,std_all_purchases,mean_all_purchases_units,std_all_purchases_units,mean_first_purchases,std_first_purchases,mean_first_purchases_units,std_first_purchases_units,total_spend,n_active_channels
organisation_id,,,,,,,,,,,,,,
7569a6a9c156a0f9398fa6cfd51df5bb,Food & Drink,2021-10-15,2024-01-13,821,298.545676,144.486937,2294.281364,867.844955,204.393423,117.681992,1245.370280,593.080210,8184572.03,9.0
784d6aa3cda59f59f2400332b2420a49,Apparel,2020-04-21,2024-03-05,1415,142.019788,56.471015,241.470671,103.179231,84.074912,35.040712,137.853710,58.980259,6309797.08,8.0
4a762f02ca755b22d37393e8dbeab1a6,Apparel,2021-07-26,2024-05-19,1029,102.275996,68.113287,172.788144,130.688970,92.617104,61.627483,151.168124,113.572514,2661171.54,9.0
ba773ebd7ec0a08f1d042187d086ccb4,Business & Industrial,2020-01-16,2024-06-02,1600,168.785000,123.386097,244.207500,176.493543,144.013750,105.433747,203.247500,147.170006,2344112.59,6.0
429c8d00704a9ef6307b49f22d5dfade,Business & Industrial,2021-02-13,2024-05-20,1193,171.962280,133.095217,243.115675,177.961038,145.108969,113.307896,199.155909,146.462215,2035450.85,6.0
f7e5ba45e3e337e8e90787ae0218a7ac,Home & Garden,2022-07-03,2024-06-02,701,17.985735,10.754392,442.540656,368.193443,13.144080,7.912237,305.259629,264.696864,733843.82,6.0
0d1fc3f1715b0c65776780e7ad8ac7df,Beauty & Fitness,2020-01-17,2021-09-01,594,13.240741,23.590612,18.811448,28.673274,11.656566,21.104891,16.464646,25.631378,520921.82,2.0
246b6e2ff861fde47d3bc5f5fdae92a8,Arts & Entertainment,2022-04-28,2024-05-20,754,37.432361,19.447107,50.584881,28.019417,29.968170,15.922590,38.228117,21.125464,391650.77,5.0
058b38dc1b5a87a8d9338196cd024d47,NaN,2022-03-30,2023-06-30,458,16.770742,8.344986,36.072052,27.853030,12.733624,6.763511,24.844978,19.995842,253944.79,8.0


 ## Why these two candidates?

 10 US brands in the dataset. Filtered to higher-spending brands for modeling
 viability. Top 5 by total spend: #1 food & drink, #2-3 apparel, #4-5 business
 & industrial. Selected the two apparel brands (apparel_1, apparel_2) as
 candidates because they share a vertical (controlled comparison) and both have
 substantial multi-channel spend.

In [ ]:
candidates = {
    "apparel_1": "784d6aa3cda59f59f2400332b2420a49",
    "apparel_2": "4a762f02ca755b22d37393e8dbeab1a6",
}

for label, org_id in candidates.items():
    brand = df[df["organisation_id"] == org_id]
    dates = brand["date_day"].sort_values()

    # Date gaps
    gaps = dates.diff().dt.days
    max_gap = gaps.max()
    n_gaps = (gaps > 1).sum()

    # Per-channel spend
    channel_spend = brand[cols_spend].sum().sort_values(ascending=False)
    active = channel_spend[channel_spend > 0]

    print(f"\n--- {label} ({org_id[:8]}...) ---")
    print(f"Date range: {dates.min().date()} to {dates.max().date()}")
    print(f"Days: {len(dates)}, Gaps >1 day: {n_gaps}, Max gap: {max_gap} days")
    print(f"Active channels ({len(active)}/9):")
    for ch, spend in active.items():
        pct_nonzero = (brand[ch] > 0).mean() * 100
        print(f"  {ch}: ${spend:,.0f} total, {pct_nonzero:.0f}% of days nonzero")


--- apparel_1 (784d6aa3...) ---
Date range: 2020-04-21 to 2024-03-05
Days: 1415, Gaps >1 day: 0, Max gap: 1.0 days
Active channels (8/9):
  meta_facebook_spend: $3,776,219 total, 90% of days nonzero
  meta_instagram_spend: $2,052,002 total, 64% of days nonzero
  google_pmax_spend: $305,338 total, 15% of days nonzero
  google_shopping_spend: $95,853 total, 13% of days nonzero
  google_display_spend: $55,449 total, 51% of days nonzero
  google_paid_search_spend: $13,576 total, 32% of days nonzero
  meta_other_spend: $5,928 total, 62% of days nonzero
  google_video_spend: $5,432 total, 5% of days nonzero

--- apparel_2 (4a762f02...) ---
Date range: 2021-07-26 to 2024-05-19
Days: 1029, Gaps >1 day: 0, Max gap: 1.0 days
Active channels (9/9):
  meta_facebook_spend: $928,447 total, 100% of days nonzero
  meta_instagram_spend: $745,782 total, 65% of days nonzero
  google_pmax_spend: $674,940 total, 72% of days nonzero
  google_paid_search_spend: $232,125 total, 95% of days nonzero
  google_s

 ## Brand selection

 **apparel_2** is the primary modeling brand. 9 active spend channels with 4
 carrying meaningful, consistent spend: Meta Facebook (100% nonzero), Google
 Paid Search (95%), Google PMax (72%), Meta Instagram (65%). This gives a real
 Google-vs-Meta allocation story ($907K Google vs. $1.67M Meta), which is
 critical for budget optimization to have something to say. 147 weeks
 (~2.8 years), zero date gaps, USD currency.

 **apparel_1** rejected because spend is too concentrated in Meta
 Facebook/Instagram ($5.8M of $6.3M total). Google channels have under 15%
 nonzero days. Not enough variation to identify channel-level effects across
 platforms.

 **Spend** (not impressions) used as the media input variable — the conventional
 choice in MMM practice (PyMC-Marketing, LightweightMMM, Robyn all default to
 spend). The tradeoff: spend conflates CPM variation with volume; impressions
 capture reach more directly. Documented but does not change the decision here.

In [ ]:
selected_orgs = [
    "784d6aa3cda59f59f2400332b2420a49",  # apparel_1
    "4a762f02ca755b22d37393e8dbeab1a6",  # apparel_2
]
df = df[df["organisation_id"].isin(selected_orgs)]

df["iso_year"] = df["date_day"].dt.isocalendar().year.astype(int)
df["iso_week"] = df["date_day"].dt.isocalendar().week.astype(int)

# Metadata cols are constant within a timeseries — take first
cols_meta = [
    "organisation_id", "organisation_vertical", "organisation_subvertical",
    "organisation_marketing_sources", "organisation_primary_territory_name",
    "territory_name", "currency_code",
]

cols_clicks = [c for c in df.columns if c.endswith("_clicks")]
cols_impressions = [c for c in df.columns if c.endswith("_impressions")]
cols_outcome = [c for c in df.columns if c.startswith(("all_purchases", "first_purchases"))]
cols_numeric = cols_spend + cols_clicks + cols_impressions + cols_outcome
agg_dict = {c: "sum" for c in cols_numeric}
agg_dict["date_day"] = "count"
agg_dict.update({c: "first" for c in cols_meta})

df_weekly = (
    df.groupby(["mmm_timeseries_id", "iso_year", "iso_week"])
    .agg(agg_dict)
    .rename(columns={"date_day": "n_days"})
    .reset_index()
)

# Drop partial weeks
n_before = len(df_weekly)
df_weekly = df_weekly[df_weekly["n_days"] == 7]
print(f"Dropped {n_before - len(df_weekly)} partial weeks")
print(f"Daily shape:  {df.shape}")
print(f"Weekly shape: {df_weekly.shape}")
df_weekly.head(5)

Dropped 2 partial weeks
Daily shape:  (2444, 52)
Weekly shape: (348, 52)


,mmm_timeseries_id,iso_year,iso_week,google_paid_search_spend,google_shopping_spend,google_pmax_spend,google_display_spend,google_video_spend,meta_facebook_spend,meta_instagram_spend,...,all_purchases_original_price,all_purchases_gross_discount,n_days,organisation_id,organisation_vertical,organisation_subvertical,organisation_marketing_sources,organisation_primary_territory_name,territory_name,currency_code
0,0349b43cddf6e720b5d582afc8cd7fd6,2021,30,163.87,678.19,0.0,0.0,0.0,683.33,0.0,...,7521.0,359.55,7,4a762f02ca755b22d37393e8dbeab1a6,Apparel,Undergarments,"Google, Meta, Tiktok",US,US,USD
1,0349b43cddf6e720b5d582afc8cd7fd6,2021,31,319.75,1451.51,0.0,0.0,0.0,1420.27,0.0,...,14174.0,566.30,7,4a762f02ca755b22d37393e8dbeab1a6,Apparel,Undergarments,"Google, Meta, Tiktok",US,US,USD
2,0349b43cddf6e720b5d582afc8cd7fd6,2021,32,349.79,1904.17,0.0,0.0,0.0,1893.06,0.0,...,18388.0,778.44,7,4a762f02ca755b22d37393e8dbeab1a6,Apparel,Undergarments,"Google, Meta, Tiktok",US,US,USD
3,0349b43cddf6e720b5d582afc8cd7fd6,2021,33,291.21,2145.83,0.0,0.0,0.0,1488.72,0.0,...,17084.0,775.58,7,4a762f02ca755b22d37393e8dbeab1a6,Apparel,Undergarments,"Google, Meta, Tiktok",US,US,USD
4,0349b43cddf6e720b5d582afc8cd7fd6,2021,34,243.88,2786.92,0.0,0.0,0.0,1075.39,0.0,...,18066.0,785.00,7,4a762f02ca755b22d37393e8dbeab1a6,Apparel,Undergarments,"Google, Meta, Tiktok",US,US,USD


 ## Aggregation notes

 Dropped 2 partial weeks (first and last weeks with fewer than 7 days). Partial
 weeks would distort weekly totals, especially spend and purchase sums, making
 them incomparable to full weeks. Both brands aggregated to keep the pipeline
 general for potential cross-brand comparison in Act 3, even though apparel_2 is
 the primary modeling brand.

In [ ]:
# Reconstruct week-start date from ISO year/week
df_weekly["week_start"] = pd.to_datetime(
    df_weekly["iso_year"].astype(str)
    + "-W"
    + df_weekly["iso_week"].astype(str).str.zfill(2)
    + "-1",
    format="%G-W%V-%u",
)
df_weekly = df_weekly.sort_values(["organisation_id", "week_start"]).reset_index(
    drop=True
)

ORG_APPAREL_2 = "4a762f02ca755b22d37393e8dbeab1a6"
df_a2 = df_weekly[df_weekly["organisation_id"] == ORG_APPAREL_2]

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08)
fig.add_trace(
    go.Scatter(x=df_a2["week_start"], y=df_a2["all_purchases"], name="All Purchases"),
    row=1, col=1,
)
fig.add_trace(
    go.Scatter(
        x=df_a2["week_start"], y=df_a2["first_purchases"], name="First Purchases"
    ),
    row=2, col=1,
)
fig.update_layout(title="apparel_2: Weekly Purchases", height=500)
fig.show()

In [ ]:
cols_spend_a2 = [c for c in cols_spend if df_a2[c].sum() > 0]

fig = go.Figure()
for ch in cols_spend_a2:
    fig.add_trace(
        go.Scatter(x=df_a2["week_start"], y=df_a2[ch], name=ch.replace("_spend", ""))
    )
fig.update_layout(title="apparel_2: Weekly Spend by Channel", height=500)
fig.show()

In [ ]:
rows = []
for ch in cols_spend_a2:
    total = df_a2[ch].sum()
    pct_zero = (df_a2[ch] == 0).mean() * 100
    nonzero = df_a2.loc[df_a2[ch] > 0, ch]
    rows.append({
        "channel": ch.replace("_spend", ""),
        "total_spend": total,
        "pct_zero_weeks": round(pct_zero, 1),
        "mean_nonzero": round(nonzero.mean(), 2) if len(nonzero) > 0 else 0,
        "median_nonzero": round(nonzero.median(), 2) if len(nonzero) > 0 else 0,
    })
df_zero = pd.DataFrame(rows).sort_values("total_spend", ascending=False)
df_zero # type: ignore

,channel,total_spend,pct_zero_weeks,mean_nonzero,median_nonzero
5,meta_facebook,928447.12,0.0,6315.97,5777.21
6,meta_instagram,745782.05,35.4,7850.34,5158.18
2,google_pmax,674940.20,27.9,6367.36,4884.66
0,google_paid_search,232124.71,2.0,1611.98,1227.84
1,google_shopping,70051.63,50.3,959.61,411.99
8,tiktok,8788.07,91.8,732.34,698.00
3,google_display,600.88,95.2,85.84,49.89
4,google_video,300.37,98.0,100.12,115.52
7,meta_other,136.51,38.1,1.50,0.84


 ## Channel assessment

 4 channels have clearly consistent spend: Meta Facebook (100% nonzero weeks),
 Google Paid Search (95%), Google PMax (72%), Meta Instagram (65%). Google
 Shopping (41% nonzero, $70K total) is borderline -- sparse but not negligible,
 and borderline significant in OLS (p=0.098). Worth including in the Bayesian
 model and letting the posterior decide. The rest are effectively
 unidentifiable: TikTok (6% nonzero), Google Display (4%), Google Video (1%),
 meta_other negligible. All channels will be included in the model, but the
 sparse ones should get wide posteriors, which is the correct behavior.

 Notable patterns in the spend time series: a large spike in Meta Facebook
 spend at the beginning and end of the data. The late spike coincides with a
 spike in purchases. All purchases and first purchases track almost
 identically, suggesting limited repeat-purchase signal in this brand. The
 discount columns in the dataset might help separate acquisition from
 promotion-driven revenue.

In [ ]:
# Time index per brand (data already sorted by org + week_start)
df_weekly["t"] = df_weekly.groupby("organisation_id").cumcount()

# Fourier features: 2 harmonic pairs at yearly frequency (52-week period)
for k in [1, 2]:
    df_weekly[f"sin_{k}"] = np.sin(2 * np.pi * k * df_weekly["t"] / 52)
    df_weekly[f"cos_{k}"] = np.cos(2 * np.pi * k * df_weekly["t"] / 52)

# Non-paid traffic aggregate
cols_nonpaid = [
    "direct_clicks", "branded_search_clicks", "organic_search_clicks",
    "email_clicks", "referral_clicks", "all_other_clicks",
]
df_weekly["nonpaid_clicks"] = df_weekly[cols_nonpaid].sum(axis=1)

print("Control columns added: t, sin_1, cos_1, sin_2, cos_2, nonpaid_clicks")
df_weekly[["week_start", "t", "sin_1", "cos_1", "sin_2", "cos_2", "nonpaid_clicks"]].head(10)

Control columns added: t, sin_1, cos_1, sin_2, cos_2, nonpaid_clicks


,week_start,t,sin_1,cos_1,sin_2,cos_2,nonpaid_clicks
0,2021-07-26,0,0.000000,1.000000,0.000000,1.000000,964.0
1,2021-08-02,1,0.120537,0.992709,0.239316,0.970942,1350.0
2,2021-08-09,2,0.239316,0.970942,0.464723,0.885456,2231.0
3,2021-08-16,3,0.354605,0.935016,0.663123,0.748511,1573.0
4,2021-08-23,4,0.464723,0.885456,0.822984,0.568065,1856.0
5,2021-08-30,5,0.568065,0.822984,0.935016,0.354605,1277.0
6,2021-09-06,6,0.663123,0.748511,0.992709,0.120537,1380.0
7,2021-09-13,7,0.748511,0.663123,0.992709,-0.120537,1651.0
8,2021-09-20,8,0.822984,0.568065,0.935016,-0.354605,1678.0
9,2021-09-27,9,0.885456,0.464723,0.822984,-0.568065,1277.0


 ## Control feature choices

 2 Fourier harmonic pairs at a 52-week period. The first harmonic captures the
 main annual cycle; the second captures asymmetry (e.g., a sharper holiday peak
 vs. a gradual summer trough). A third harmonic would model ~17-week
 sub-cycles, which risks overfitting on only 147 weeks of data. Non-paid
 traffic clicks aggregated into a single feature rather than kept as 6 separate
 regressors. With 147 observations and 9 spend channels already in the model,
 splitting nonpaid into separate channels would eat degrees of freedom for
 variables that are not the focus of the analysis.

In [ ]:
df_ols = df_weekly[df_weekly["organisation_id"] == ORG_APPAREL_2].copy()

# Only include spend channels with nonzero total
cols_spend_active = [c for c in cols_spend if df_ols[c].sum() > 0]
print(f"Active spend channels: {cols_spend_active}")

y = df_ols["all_purchases"]

# Model A: spend channels only
X_a = sm.add_constant(df_ols[cols_spend_active])
model_a = sm.OLS(y, X_a).fit()
print(f"\n--- Model A: Spend only (R² = {model_a.rsquared:.3f}) ---")
print(model_a.summary().tables[1])

# Model B: spend + trend + seasonality
cols_controls = ["t", "sin_1", "cos_1", "sin_2", "cos_2"]
X_b = sm.add_constant(df_ols[cols_spend_active + cols_controls])
model_b = sm.OLS(y, X_b).fit()
print(f"\n--- Model B: Spend + trend/seasonality (R² = {model_b.rsquared:.3f}) ---")
print(model_b.summary().tables[1])

# Model C: + all non-paid traffic
X_c = sm.add_constant(df_ols[cols_spend_active + cols_controls + ["nonpaid_clicks"]])
model_c = sm.OLS(y, X_c).fit()
print(f"\n--- Model C: + all nonpaid_clicks (R² = {model_c.rsquared:.3f}) ---")
print(model_c.summary().tables[1])

# Model D: + exogenous-only non-paid traffic (exclude branded_search, direct)
cols_exog_nonpaid = [
    "organic_search_clicks", "email_clicks", "referral_clicks", "all_other_clicks",
]
df_ols["exog_nonpaid_clicks"] = df_ols[cols_exog_nonpaid].sum(axis=1)
X_d = sm.add_constant(df_ols[cols_spend_active + cols_controls + ["exog_nonpaid_clicks"]])
model_d = sm.OLS(y, X_d).fit()
print(f"\n--- Model D: + exog nonpaid only (R² = {model_d.rsquared:.3f}) ---")
print(model_d.summary().tables[1])

Active spend channels: ['google_paid_search_spend', 'google_shopping_spend', 'google_pmax_spend', 'google_display_spend', 'google_video_spend', 'meta_facebook_spend', 'meta_instagram_spend', 'meta_other_spend', 'tiktok_spend']

--- Model A: Spend only (R² = 0.914) ---
                               coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------
const                      129.3832     25.326      5.109      0.000      79.303     179.463
google_paid_search_spend     0.0303      0.009      3.264      0.001       0.012       0.049
google_shopping_spend        0.0237      0.014      1.667      0.098      -0.004       0.052
google_pmax_spend            0.0117      0.006      1.950      0.053      -0.000       0.024
google_display_spend        -0.1261      0.500     -0.253      0.801      -1.114       0.862
google_video_spend           0.1618      0.858      0.188      0.851      -1.536

In [ ]:
df_compare = pd.DataFrame([
    {"model": "A", "regressors": "spend only", "R²": model_a.rsquared, "adj_R²": model_a.rsquared_adj},
    {"model": "B", "regressors": "+ trend + seasonality", "R²": model_b.rsquared, "adj_R²": model_b.rsquared_adj},
    {"model": "C", "regressors": "+ all nonpaid_clicks", "R²": model_c.rsquared, "adj_R²": model_c.rsquared_adj},
    {"model": "D", "regressors": "+ exog nonpaid only", "R²": model_d.rsquared, "adj_R²": model_d.rsquared_adj},
])
df_compare[["R²", "adj_R²"]] = df_compare[["R²", "adj_R²"]].round(3)
df_compare # type: ignore

,model,regressors,R²,adj_R²
0,A,spend only,0.914,0.908
1,B,+ trend + seasonality,0.932,0.925
2,C,+ all nonpaid_clicks,0.939,0.932
3,D,+ exog nonpaid only,0.934,0.926


 ## OLS takeaways

 R² is high across all specs (0.914 to 0.939), but this should not be mistaken
 for a good model. OLS here is capturing correlations between spend and
 purchases with no adstock (carryover effects) and no saturation (diminishing
 returns). The coefficients are not causal estimates and should not be used for
 budget decisions. Several channels show nonsensical coefficients (meta_other
 at -26, negative TikTok) driven by collinearity and sparse data.

 What the OLS exercise does confirm: (1) spend and purchases move together,
 which is a necessary baseline for modeling; (2) seasonality is real but
 incremental (R² jumps only 0.018 from Model A to B, first Fourier harmonic is
 significant); (3) nonpaid traffic adds modest explanatory power (R² 0.932 to
 0.939). This is a sanity check, not a model to make decisions from. The
 Bayesian model in Phase 2 adds adstock and saturation, which will change the
 coefficient interpretation entirely.

In [ ]:
os.makedirs("data/processed", exist_ok=True)

ORG_APPAREL_1 = "784d6aa3cda59f59f2400332b2420a49"

for label, org_id in [("apparel_1", ORG_APPAREL_1), ("apparel_2", ORG_APPAREL_2)]:
    df_brand = df_weekly[df_weekly["organisation_id"] == org_id]
    path = f"data/processed/{label}.csv"
    df_brand.to_csv(path, index=False)
    print(f"Saved {label}: {df_brand.shape} → {path}")

Saved apparel_1: (201, 59) → data/processed/apparel_1.csv
Saved apparel_2: (147, 59) → data/processed/apparel_2.csv
